# Custom CNN architectures and parameter counts

EN3150 Assignment 03 - Group 9. Task 2.

The source cells expose the standard convolutions, depthwise/pointwise blocks, activation functions and pooling heads used in the report. The final cell constructs the actual architectures, without training or downloads.

## Running this notebook

Use Python 3.10+ with the project dependencies. Extract the whole ZIP and open this notebook beside raw_code/. On Colab, upload and extract the complete ZIP before running. Set INSTALL_DEPENDENCIES = True in the setup cell for a fresh runtime. Run cells in order with a fresh kernel for each notebook.

The writefile cells below contain the original Python source, including comments. They save that code into the bundled package so relative imports and CLI entry points behave exactly as in the repository. The complete package is already included: the notebooks can be run independently.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Extract the entire ZIP first. This also works from notebooks/submission in Git.
start = Path.cwd().resolve()
candidates = [start, *start.parents]
PROJECT = next((base for parent in candidates
                for base in (parent / "raw_code", parent)
                if (base / "src/pr_edge").is_dir()
                and (base / "configs/base.yaml").is_file()), None)
if PROJECT is None:
    raise FileNotFoundError("Extract the complete code ZIP, including raw_code, first.")
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT / "src"))
sys.path.insert(0, str(PROJECT))

# Enable once in a fresh Python 3.10+ / Colab environment (requires internet).
INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)

def run_script(script, *args):
    subprocess.run([sys.executable, str(PROJECT / script), *map(str, args)],
                   cwd=PROJECT, check=True)

print("Project:", PROJECT)


## Source: `src/pr_edge/models/model_a.py`


In [ ]:
%%writefile src/pr_edge/models/model_a.py
"""Model A - the standard CNN baseline (issue #9).

Interleaved standard 2D convolutions and max-pooling, then a classifier head,
matching the topology in figure 1 of the assignment brief. This is the heavy
baseline that Model B's depthwise separable variant is measured against, so it
should stay deliberately conventional.

The template is the lecturer's own example CNN in docs/slides/CNNs.pdf p54:

    Conv2D(32, 3x3, relu) -> MaxPool 2x2 -> Conv2D(64, 3x3, relu) -> MaxPool 2x2
    -> Flatten -> Dense(128, relu) -> Dense(64, relu) -> Dense(10, softmax)

Model A keeps that shape - valid (unpadded) convolutions, no batch
normalisation - with one more conv stage for the larger 64x64x3 input. Stage
widths and dense widths come from the config, so the parameter arithmetic in
#11 can be stated from the config alone.

Model A is allowed a Flatten -> Dense head. That is part of what makes it the
heavy baseline, and the contrast with Model B's global average pooling is a
real finding for the report: as p54 itself notes, the majority of parameters
reside within the fully-connected dense layers.
"""

from __future__ import annotations

import torch
import torch.nn as nn

from ..config import ModelConfig

# 3x3 kernels and 2x2/stride-2 max pooling, as in the CNNs.pdf p54 example.
# p34 gives F = 2, S = 2 as the usual pooling choice.
KERNEL_SIZE = 3
POOL_SIZE = 2

HEADS = ("flatten", "gap")


def _conv_output_size(size: int) -> int:
    """Spatial size after one valid 3x3 convolution and one 2x2 max pool.

    CNNs.pdf p32: an unpadded K x K filter maps W to W - K + 1. Pooling then
    halves it, flooring odd sizes (29 -> 14), exactly as p54's 11 -> 5 does.
    """
    return (size - KERNEL_SIZE + 1) // POOL_SIZE


class ModelA(nn.Module):
    """Standard Conv/ReLU/MaxPool stack. Returns raw logits, never softmax.

    Args:
        cfg: model section of the run config. ``channels`` gives the width of
            each conv stage, ``fc_units`` the hidden dense widths, ``head``
            selects ``flatten`` (as in p54) or ``gap`` (global average pooling),
            and ``dropout`` is applied before each hidden dense layer.

    With the committed config (channels [32, 64, 128], fc_units [128, 64]) the
    shapes are 64 -> 62 -> 31 -> 29 -> 14 -> 12 -> 6, a 128 x 6 x 6 = 4608-wide
    flatten, and 692,106 parameters, 86.5% of them in the dense head.
    """

    # The brief caps input resolution at 64x64 RGB. These are class attributes
    # rather than constructor arguments because the frozen signature is
    # ModelA(cfg) and the model section cannot see data.input_size; the flatten
    # width must nonetheless be known before the first forward pass.
    IN_CHANNELS = 3
    INPUT_SIZE = 64

    def __init__(self, cfg: ModelConfig) -> None:
        super().__init__()
        channels = list(cfg.channels)
        fc_units = list(cfg.fc_units)
        if not channels or any(type(c) is not int or c <= 0 for c in channels):
            raise ValueError(f"model.channels must be a non-empty list of positive ints, got {channels}")
        if any(type(u) is not int or u <= 0 for u in fc_units):
            raise ValueError(f"model.fc_units must be positive ints, got {fc_units}")
        if cfg.head not in HEADS:
            raise ValueError(f"model.head must be one of {HEADS}, got {cfg.head!r}")
        if not 0 <= cfg.dropout < 1:
            raise ValueError(f"model.dropout must be in [0, 1), got {cfg.dropout}")

        # Feature extractor: conv -> ReLU -> pool per stage. p33 gives this as
        # the canonical block ordering.
        layers: list[nn.Module] = []
        size = self.INPUT_SIZE
        in_ch = self.IN_CHANNELS
        for out_ch in channels:
            layers += [
                self._convolution(in_ch, out_ch),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(kernel_size=POOL_SIZE, stride=POOL_SIZE),
            ]
            size = _conv_output_size(size)
            in_ch = out_ch
        if size < 1:
            raise ValueError(
                f"{len(channels)} conv stages shrink a {self.INPUT_SIZE}x{self.INPUT_SIZE} "
                "input to nothing; use fewer stages"
            )
        self.features = nn.Sequential(*layers)
        self.feature_size = size

        # Classifier head. "flatten" keeps every spatial position (the heavy
        # baseline); "gap" averages each channel to one value first.
        head: list[nn.Module] = []
        if cfg.head == "gap":
            head.append(nn.AdaptiveAvgPool2d(1))
            width = in_ch
        else:
            width = in_ch * size * size
        head.append(nn.Flatten())
        for units in fc_units:
            if cfg.dropout > 0:
                head.append(nn.Dropout(cfg.dropout))
            head += [nn.Linear(width, units), nn.ReLU(inplace=True)]
            width = units
        # Output layer: raw logits. Softmax belongs to CrossEntropyLoss, and
        # no dropout sits directly in front of the class scores.
        head.append(nn.Linear(width, cfg.num_classes))
        self.classifier = nn.Sequential(*head)

    def _convolution(self, in_ch: int, out_ch: int) -> nn.Module:
        """The convolution of one stage: a standard valid 3x3.

        The single point Model B overrides - it swaps this for a depthwise
        separable block and changes nothing else, which is what makes the
        A-vs-B comparison attributable to that substitution alone.
        """
        return nn.Conv2d(in_ch, out_ch, kernel_size=KERNEL_SIZE)  # valid: no padding

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """(N, 3, 64, 64) -> (N, num_classes) logits."""
        expected = (self.IN_CHANNELS, self.INPUT_SIZE, self.INPUT_SIZE)
        if tuple(x.shape[1:]) != expected:
            # Without this, a 224x224 batch fails deep inside Linear with a
            # matrix-shape message that names neither the model nor the fix.
            raise ValueError(
                f"ModelA expects input of shape (N, {expected[0]}, {expected[1]}, {expected[2]}), "
                f"got {tuple(x.shape)}. Set data.input_size to {self.INPUT_SIZE}."
            )
        return self.classifier(self.features(x))


## Source: `src/pr_edge/models/model_b.py`


In [ ]:
%%writefile src/pr_edge/models/model_b.py
"""Model B - the depthwise separable CNN (issue #10).

Model A's structure with every standard convolution replaced by a depthwise
separable block, per docs/slides/CNNs.pdf p43-p45 and the MobileNets paper.

This is a pure substitution: the stage widths ([32, 64, 128]), valid padding,
2x2 pooling, dense head (128 -> 64 -> 10) and dropout are Model A's, and only
two things change - each 3x3 convolution becomes a depthwise separable block,
and Flatten becomes global average pooling (``head: gap`` in the config). Any
difference between the two models in the report is therefore attributable to
those two levers alone.

HARD CONSTRAINT: total trainable parameters must not exceed 100,000. With the
committed config Model B has 36,968 (Model A: 692,106) and needs 3,725,196
multiply-accumulates per image (Model A: 30,038,016).

``tests/test_model_b.py`` asserts the budget and CI enforces it, so a violation
cannot be merged.
"""

from __future__ import annotations

import torch
import torch.nn as nn

from .model_a import KERNEL_SIZE, ModelA


class DepthwiseSeparableBlock(nn.Module):
    """Depthwise 3x3 followed by pointwise 1x1.

    Kept as two separately inspectable stages so the per-layer parameter and
    MAC derivation in #11 can name each one, mirroring the worked example in
    CNNs.pdf p44:

        regular 3x3, 256 -> 256 :  9 x 256 x 256  ~= 590,000 ops
        depthwise 3x3           :  9 x 256        ~=   2,300 ops
        pointwise 1x1, 256->256 :      256 x 256  ~=  65,500 ops
        total                   :                 ~=  67,800 ops  (8.7x speedup)

    p44 draws the block as those two steps with nothing between them, so there
    is no activation inside it: the block is a drop-in replacement for one
    standard convolution, and the ReLU that follows a convolution in Model A
    follows the whole block here.
    """

    def __init__(self, in_ch: int, out_ch: int, stride: int = 1) -> None:
        super().__init__()
        # Step 1: one 3x3 filter per input channel. groups=in_ch is p45's
        # group-wise convolution with G = C_in. Unpadded, like Model A, so
        # both models produce the same spatial sizes stage by stage.
        self.depthwise = nn.Conv2d(
            in_ch, in_ch, kernel_size=KERNEL_SIZE, stride=stride, groups=in_ch,
        )
        # Step 2: a 1x1 convolution mixing the channels, C_in -> C_out.
        self.pointwise = nn.Conv2d(in_ch, out_ch, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.pointwise(self.depthwise(x))


class ModelB(ModelA):
    """Lightweight CNN, <= 100k trainable parameters. Returns raw logits.

    Same input and output shapes as :class:`~pr_edge.models.model_a.ModelA` so
    the comparison is like-for-like. It inherits Model A's construction,
    validation and 64x64 input check, overriding only the per-stage
    convolution - the rest of the difference (global average pooling instead
    of Flatten) comes from ``head: gap`` in ``configs/model_b.yaml``.
    """

    def _convolution(self, in_ch: int, out_ch: int) -> nn.Module:
        return DepthwiseSeparableBlock(in_ch, out_ch)


## Source: `src/pr_edge/models/_smoke.py`


In [ ]:
%%writefile src/pr_edge/models/_smoke.py
"""A deliberately tiny network used only by smoke mode.

Not part of the assignment. It exists so that ``--smoke`` can exercise the full
config -> train -> evaluate -> profile -> artifacts path without depending on
Model A (#9), Model B (#10) or the SOTA harness (#19) being implemented yet,
and without CI spending a minute on a real architecture.

Keep it trivial. If this ever grows features, something has gone wrong.
"""

from __future__ import annotations

import torch
import torch.nn as nn

from ..config import ModelConfig


class SmokeNet(nn.Module):
    """One conv, global average pool, one linear. Returns raw logits."""

    def __init__(self, cfg: ModelConfig) -> None:
        super().__init__()
        self.conv = nn.Conv2d(3, 8, kernel_size=3, padding=1)
        self.act = nn.ReLU(inplace=True)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(8, cfg.num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.act(self.conv(x))
        x = self.pool(x).flatten(1)
        return self.fc(x)


## Source: `src/pr_edge/models/__init__.py`


In [ ]:
%%writefile src/pr_edge/models/__init__.py
"""Model registry.

``build_model`` is the single entry point every script uses, so adding an
architecture means adding a registry entry, not editing the training loop.
"""

from __future__ import annotations

import torch.nn as nn

from ..config import Config
from ._smoke import SmokeNet
from .model_a import ModelA
from .model_b import ModelB
from .sota import build_sota

__all__ = ["build_model", "ModelA", "ModelB", "build_sota", "SmokeNet"]

_CUSTOM = {"model_a": ModelA, "model_b": ModelB}
_SOTA = {"mobilenet_v2", "efficientnet_b0"}


def build_model(cfg: Config) -> nn.Module:
    """Construct the model named by ``cfg.model.name``.

    In smoke mode the named architecture is bypassed entirely in favour of a
    tiny stand-in. That keeps #1 from having to implement #9, #10 and #19 just
    to make CI run, and it keeps the smoke test fast: the point is to exercise
    the config-to-artifacts wiring, not the architecture.
    """
    if cfg.training.smoke:
        return SmokeNet(cfg.model)

    name = cfg.model.name
    if name in _CUSTOM:
        return _CUSTOM[name](cfg.model)
    if name in _SOTA:
        return build_sota(cfg)

    known = sorted(_CUSTOM) + sorted(_SOTA)
    raise ValueError(f"unknown model '{name}'. Known models: {known}")


## Run / inspect results


In [ ]:
import torch
from pr_edge.config import load_config
from pr_edge.models import build_model
from pr_edge.device import set_seed

for name in ("model_a", "model_b"):
    cfg = load_config(PROJECT / "configs" / f"{name}.yaml")
    set_seed(cfg.seed)
    model = build_model(cfg).eval()
    with torch.no_grad():
        output = model(torch.zeros(2, 3, 64, 64))
    count = sum(parameter.numel() for parameter in model.parameters())
    print(name, "parameters:", count, "output shape:", tuple(output.shape))
    if name == "model_b":
        assert count <= 100_000
